# 05b – AutoML met PyCaret

| | |
|---|---|
| **Auteur(s)** | Kobe Vandenberk |
| **Wat** | Een brede, automatische vergelijking van regressiemodellen met PyCaret, als controle op onze eigen modelkeuzes (05c–05e). |
| **Input** | `data/2024/model_table.parquet` |
| **Output** | `models/2024/metrics/pycaret.json`, `models/2024/predictions/pycaret.parquet` |
| **GenAI** | Code en tekst opgesteld met hulp van Claude; keuzes en resultaten nagekeken door Kobe. |

> **Studie op 2024 (modelselectie).** Dit notebook werkt op één jaar (2024) om snel en grondig te kiezen welke techniek het best werkt. De gekozen techniek wordt daarna op alle data (2013–2026) getraind: zie `02b_eda_all_years`, `04_prepare_data` en `05e_hist_gradient_boosting`.

## Waarom AutoML?
We kozen zelf drie modelfamilies (GLM, random forest, gradient boosting). AutoML is een **onafhankelijke controle**: probeert PyCaret iets wat wij over het hoofd zagen dat veel beter werkt?

Om eerlijk te vergelijken, geven we PyCaret **dezelfde split en dezelfde validatie**:
- de trainset gaat naar `setup`, de testset als `test_data` (PyCaret maakt dus geen eigen willekeurige split);
- cross-validatie met `GroupKFold` op kalenderweek (`fold_strategy="groupkfold"`, `fold_groups="week"`);
- sorteren op **MAE**.

PyCaret 3.3 werkt enkel op Python ≤ 3.11; daarom gebruiken we de `venv_pycaret`-omgeving.

**Welke modellen, en waarom niet alle?** Op 200.000 rijen schalen niet alle modellen van PyCaret:
- **uitgesloten**: `svm` en `kr` (kernelmethodes, kwadratisch in het aantal rijen), `gpr` (Gaussian process), `knn` (voorspellen = afstand tot alle trainrijen), `mlp`, `ada`, `ransac`, `tr`, `par`, `lar`, `llar`, `omp`. Ze zijn traag of zijn varianten van lineaire regressie die hier niets toevoegen;
- **meegenomen**: boosting (`lightgbm`, `xgboost`, `gbr`), bagging (`rf`, `et`), één boom (`dt`), lineaire modellen (`lr`, `ridge`, `lasso`, `en`, `br`, `huber`) en de `dummy`-baseline. De sterkste kandidaten staan vooraan, en `compare_models` krijgt een **tijdsbudget** van 25 minuten.

**Les uit een eerste poging:** met `n_jobs=-1` liep de eerste run na een uur nog altijd. PyCaret draait de folds parallel *en* geeft elk model `n_jobs=-1`, waardoor 22 processen elk alle 22 kernen probeerden te gebruiken (*oversubscription*): de processen zaten vooral op elkaar te wachten. Met `n_jobs=4` blijft het aantal gelijktijdige threads beperkt.

## 0. Imports en data

In [1]:
import pandas as pd
from pycaret.regression import compare_models, predict_model, pull, setup, tune_model

import citibike as cb

table = cb.load_model_table(cb.DATA_2024 / "model_table.parquet")
X_train, y_train, g_train, X_test, y_test = cb.train_test(table, cb.FEATURES_2024)
train_df = X_train.assign(trips=y_train.to_numpy(), week=g_train.to_numpy())
test_df = X_test.assign(trips=y_test.to_numpy(), week=0)
print(train_df.shape, test_df.shape)

(203040, 11) (60480, 11)


## 1. Setup
- `zone` opgegeven als categorie (`categorical_features`). **Let op:** PyCaret meldt hieronder *"No categorical columns found"* en houdt 9 kolommen over: de zone (een geheel getal) werd toch als **getal** behandeld. Zie de interpretatie onder de vergelijking;
- `week` is alleen de groep voor de folds, geen kenmerk (`ignore_features`);
- geen extra normalisatie of transformaties: we willen weten hoe de modellen op dezelfde kenmerken presteren.

In [2]:
exp = setup(data=train_df, test_data=test_df, target="trips", session_id=cb.SEED,
            categorical_features=["zone"], ignore_features=["week"],
            numeric_features=[c for c in cb.FEATURES_2024 if c != "zone"],
            fold_strategy="groupkfold", fold_groups="week", fold=5,
            html=False, verbose=False, n_jobs=4)
print("Kolommen na preprocessing:", exp.get_config("X_train_transformed").shape[1])

Kolommen na preprocessing: 9


## 2. Alle modellen vergelijken

In [3]:
top3 = compare_models(sort="MAE", n_select=3, budget_time=25, verbose=False,
                      include=["lightgbm", "xgboost", "et", "rf", "gbr", "dt", "huber", "br", "ridge", "lasso", "en", "lr", "dummy"])
leaderboard = pull()
leaderboard

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
et,Extra Trees Regressor,26.9570,2388.0412,48.6290,0.9317,0.3810,0.3462,3.772
rf,Random Forest Regressor,28.1429,2680.6037,51.5001,0.9235,0.3947,0.3659,5.508
lightgbm,Light Gradient Boosting Machine,30.7167,2629.1721,51.2190,0.9257,0.4572,0.4522,0.666
xgboost,Extreme Gradient Boosting,31.1993,2806.3531,52.8560,0.9201,0.5048,0.4873,0.210
dt,Decision Tree Regressor,36.2355,4409.2101,66.2374,0.8746,0.4822,0.4356,0.348
gbr,Gradient Boosting Regressor,47.0773,6103.5735,78.1124,0.8288,0.6490,0.7874,3.572
huber,Huber Regressor,93.4384,20385.2602,142.6795,0.4302,1.1146,2.5209,1.030
lasso,Lasso Regression,97.0771,19213.3191,138.5602,0.4620,1.1958,3.1255,0.120
en,Elastic Net,97.0930,19266.7456,138.7535,0.4604,1.1946,3.1107,0.112
br,Bayesian Ridge,97.2238,19200.6689,138.5170,0.4623,1.1967,3.1465,0.092


**Interpretatie:**
- **Bomen-ensembles winnen**: Extra Trees (CV-MAE 27,0), random forest (28,1), LightGBM (30,7) en XGBoost (31,2). Dezelfde rangorde van modelfamilies als in onze eigen notebooks.
- **Lineaire modellen falen volledig** (MAE ± 97, R² 0,46), en dat is leerrijk. Ze krijgen hier de ruwe kolommen: `hour` als één getal (dus "de vraag stijgt lineair van 0u tot 23u") en `zone` als getal, zonder interacties. Ons eigen Poisson-GLM met zone × uur × weekend haalde 30,5 (05c). Het verschil tussen 97 en 30 is volledig **feature engineering**: een lineair model is maar zo goed als de kenmerken die je het geeft.
- Dat de zone als getal behandeld werd, is voor bomen een klein probleem: de zones zijn genummerd van druk naar rustig (04), dus een split "zone ≤ 5" is zinvol (zoals bij de random forest in 05d).
- Het hele vergelijken duurde maar 2 minuten (zie `TT (Sec)`: tijd per fold).

## 3. De top 3 tunen
`tune_model` doet een random search (15 combinaties) met dezelfde folds, geoptimaliseerd op MAE. `choose_better=True` houdt het ongetunede model als tunen niets oplevert.

**Eigen zoekruimte voor random forest en extra trees.** PyCaret's standaard-zoekruimte voor die twee bevat `criterion="absolute_error"`. Een boom die op de absolute fout splitst, moet bij elke mogelijke split een mediaan herberekenen; op 160.000 rijen is dat tientallen keren trager. Een eerste run bleef daardoor meer dan een uur hangen op het tunen van het eerste model. We geven die modellen daarom een eigen zoekruimte met het gewone `squared_error`-criterium, rond de instellingen die in 05d goed werkten. LightGBM gebruikt de standaardruimte van PyCaret.

In [4]:
tuned, tuning_rows = [], []
TREE_GRID = {"n_estimators": [100, 200, 300], "max_features": [0.6, 0.8, 1.0],
             "min_samples_leaf": [1, 2, 4], "max_depth": [None, 20, 30]}
for model in top3:
    grid = TREE_GRID if type(model).__name__ in ("ExtraTreesRegressor", "RandomForestRegressor") else None
    t = tune_model(model, n_iter=15, optimize="MAE", choose_better=True, custom_grid=grid, verbose=False)
    scores = pull()
    tuned.append(t)
    tuning_rows.append({"model": type(model).__name__, "cv_mae_getuned": round(scores.loc["Mean", "MAE"], 4)})
pd.DataFrame(tuning_rows)

,model,cv_mae_getuned
0,ExtraTreesRegressor,26.6368
1,RandomForestRegressor,27.5106
2,LGBMRegressor,29.3975


## 4. Eindtest van het beste PyCaret-model

In [5]:
best_cv = min(range(len(tuned)), key=lambda i: tuning_rows[i]["cv_mae_getuned"])
best = tuned[best_cv]
holdout = predict_model(best, verbose=False)        # zonder data = de test_data uit setup
pred = holdout["prediction_label"].to_numpy()
test_metrics = cb.evaluate(y_test, pred)
metrics = {"notebook": "05b_pycaret", "model": f"PyCaret: {type(best).__name__} (getuned)",
           "leaderboard": leaderboard[["Model", "MAE", "RMSE", "R2", "TT (Sec)"]].head(10).to_dict("records"),
           "tuning": tuning_rows, "cv_mae_mean": tuning_rows[best_cv]["cv_mae_getuned"],
           **{f"test_{k}": v for k, v in test_metrics.items()}}
cb.save_results("pycaret", metrics, table.loc[y_test.index, ["zone", "time", "trips"]].assign(pred=pred),
                cb.METRICS_2024, cb.PRED_2024)
pd.Series(test_metrics)

mae                 30.5775
rmse                59.9382
r2                   0.8932
poisson_deviance    14.5678
dtype: float64

**Interpretatie:** tunen helpt alle drie een beetje: Extra Trees 27,0 naar 26,6, random forest 28,1 naar 27,5, LightGBM 30,7 naar 29,4. Het beste PyCaret-model (Extra Trees) haalt op de testset MAE 30,6 en R² 0,893.

## Samenvatting
| | CV-MAE | Test-MAE | Test-R² |
|---|---|---|---|
| PyCaret: Extra Trees (getuned) | 26,6 | 30,6 | 0,893 |
| ter vergelijking, eigen getunede HistGradientBoosting (05e) | 24,9 | 29,0 | 0,906 |

- AutoML vindt **geen beter model** dan ons eigen getunede boosting-model. Het bevestigt wel de keuze voor boom-ensembles.
- Waarom doet onze HistGradientBoosting het beter dan PyCaret's LightGBM (een vergelijkbaar algoritme)? Wij gebruikten een **Poisson-verliesfunctie** (passend bij tellingen), behandelden de zone als **echte categorie**, en tunden met meer bomen en een kleinere learning rate.
- Twee lessen over AutoML: het is niet gratis (zie de valkuilen hierboven: *oversubscription* en een trage zoekruimte), en je moet controleren wat het met je kenmerken doet (de zone werd stilzwijgend een getal).